# Xcapit FHE-ML Platform - SDK Real con TenSEAL

## Demo de Encriptacion Homomorfica REAL

### A diferencia de los otros notebooks...
Este demo usa **encriptacion FHE real** con la libreria TenSEAL, no simulaciones.

### Que vas a ver:
1. Creacion de contexto CKKS con parametros de seguridad reales
2. Encriptacion de datos con llaves criptograficas
3. Operaciones matematicas sobre datos cifrados
4. Entrenamiento de modelo sobre ciphertext
5. Comparacion de precision: plaintext vs FHE
6. Benchmark de tiempos de ejecucion

---

## Flujo del Demo (Paso a Paso)

```
+------------------+     +------------------+     +------------------+
|   PASO 1-2       |     |   PASO 3-4       |     |   PASO 5-6       |
|   Setup +        | --> |   Contexto       | --> |   Encriptacion   |
|   TenSEAL        |     |   CKKS           |     |   Real           |
+------------------+     +------------------+     +------------------+
         |                        |                        |
         v                        v                        v
+------------------+     +------------------+     +------------------+
|   PASO 7         |     |   PASO 8         |     |   PASO 9         |
|   Operaciones    | --> |   ML sobre       | --> |   Benchmark      |
|   Homomorficas   |     |   Ciphertext     |     |   Rendimiento    |
+------------------+     +------------------+     +------------------+
```

### Que aprenderas:
- Como funciona CKKS internamente
- Que operaciones puedes hacer sobre datos encriptados
- Cual es el overhead de FHE vs plaintext
- Cuando vale la pena usar FHE

---

### Requisitos

| Libreria | Version | Instalacion |
|----------|---------|-------------|
| TenSEAL | >= 0.3.0 | `pip install tenseal` |
| scikit-learn | >= 1.0 | `pip install scikit-learn` |
| numpy | >= 1.20 | `pip install numpy` |

### Parametros CKKS Explicados

| Parametro | Que hace | Valor tipico |
|-----------|----------|--------------|
| `poly_modulus_degree` | Precision de operaciones | 8192 o 16384 |
| `coeff_mod_bit_sizes` | Niveles de multiplicacion | [60, 40, 40, 60] |
| `global_scale` | Escala de numeros | 2^40 |
| `security_level` | Bits de seguridad | 128, 192, 256 |

### Trade-offs de FHE

| Aspecto | Plaintext | FHE |
|---------|-----------|-----|
| Velocidad | Instantaneo | 100-1000x mas lento |
| Tamano datos | 1x | ~100x mas grande |
| Precision | Exacta | Aproximada (CKKS) |
| Privacidad | Sin proteccion | Maxima proteccion |
| Uso recomendado | Datos no sensibles | PHI, PII, secretos |

## 1. Setup e Imports

In [ ]:
import sys
import os
import time
import numpy as np
import pandas as pd
from datetime import datetime

# Verificar TenSEAL
try:
    import tenseal as ts
    TENSEAL_AVAILABLE = True
    print(f"TenSEAL version: {ts.__version__}")
except ImportError:
    TENSEAL_AVAILABLE = False
    print("ADVERTENCIA: TenSEAL no esta instalado. Instalar con: pip install tenseal")

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

print("\nXcapit FHE-ML Platform - SDK Real Demo")
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)

## 2. Creacion del Contexto CKKS

El esquema CKKS permite operaciones aritmeticas aproximadas sobre numeros reales encriptados.

In [ ]:
def create_ckks_context(security_level=128, poly_modulus_degree=8192):
    """
    Crea un contexto CKKS con los parametros especificados.
    
    Args:
        security_level: 128, 192, o 256 bits
        poly_modulus_degree: Grado del polinomio (potencia de 2)
        
    Returns:
        TenSEAL context configurado
    """
    if not TENSEAL_AVAILABLE:
        print("TenSEAL no disponible - retornando None")
        return None
    
    # Configurar coef_mod_bit_sizes segun nivel de seguridad
    if security_level == 128:
        coef_mod_bit_sizes = [60, 40, 40, 60]  # Para poly_degree 8192
    elif security_level == 192:
        coef_mod_bit_sizes = [60, 40, 40, 40, 60]
    else:  # 256
        coef_mod_bit_sizes = [60, 40, 40, 40, 40, 60]
    
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=poly_modulus_degree,
        coeff_mod_bit_sizes=coef_mod_bit_sizes
    )
    
    # Generar claves
    context.generate_galois_keys()
    context.global_scale = 2**40
    
    return context

# Crear contexto
if TENSEAL_AVAILABLE:
    print("Creando contexto CKKS...")
    start_time = time.time()
    
    ctx = create_ckks_context(security_level=128, poly_modulus_degree=8192)
    
    elapsed = time.time() - start_time
    print(f"Contexto creado en {elapsed:.2f} segundos")
    print(f"\nParametros:")
    print(f"  Esquema: CKKS")
    print(f"  Seguridad: 128 bits")
    print(f"  Poly modulus degree: 8192")
    print(f"  Scale: 2^40")
else:
    ctx = None
    print("Contexto no creado - TenSEAL no disponible")

## 3. Generacion de Datos de Prueba

In [ ]:
# Generar datos sinteticos para clasificacion binaria
np.random.seed(42)

# Usar menos muestras para FHE (es computacionalmente intensivo)
N_SAMPLES = 500  # Reducido para demo
N_FEATURES = 10

X, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=7,
    n_redundant=2,
    n_classes=2,
    random_state=42
)

# Normalizar datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset generado:")
print(f"  Total muestras: {N_SAMPLES}")
print(f"  Features: {N_FEATURES}")
print(f"  Train: {len(X_train)} muestras")
print(f"  Test: {len(X_test)} muestras")
print(f"  Clase positiva: {y.sum()} ({y.mean()*100:.1f}%)")

## 4. Encriptacion de Datos con CKKS

In [ ]:
def encrypt_data(context, data):
    """
    Encripta un array numpy usando CKKS.
    
    Args:
        context: Contexto TenSEAL
        data: numpy array (2D)
        
    Returns:
        Lista de vectores CKKS encriptados
    """
    if context is None:
        return None
    
    encrypted_vectors = []
    for row in data:
        enc_vector = ts.ckks_vector(context, row.tolist())
        encrypted_vectors.append(enc_vector)
    
    return encrypted_vectors

def decrypt_data(encrypted_vectors):
    """
    Desencripta vectores CKKS.
    
    Args:
        encrypted_vectors: Lista de vectores CKKS
        
    Returns:
        numpy array con datos desencriptados
    """
    if encrypted_vectors is None:
        return None
    
    decrypted = [vec.decrypt() for vec in encrypted_vectors]
    return np.array(decrypted)

# Encriptar datos de entrenamiento
if TENSEAL_AVAILABLE and ctx is not None:
    print("Encriptando datos de entrenamiento...")
    start_time = time.time()
    
    X_train_enc = encrypt_data(ctx, X_train)
    
    elapsed = time.time() - start_time
    print(f"Encriptacion completada en {elapsed:.2f} segundos")
    print(f"  Vectores encriptados: {len(X_train_enc)}")
    
    # Mostrar tamano del ciphertext
    sample_size = len(X_train_enc[0].serialize())
    total_size = sample_size * len(X_train_enc) / 1024 / 1024
    print(f"  Tamano por vector: {sample_size / 1024:.2f} KB")
    print(f"  Tamano total cifrado: {total_size:.2f} MB")
else:
    X_train_enc = None
    print("Encriptacion no disponible - TenSEAL no instalado")

## 5. Operaciones Homomorficas Basicas

In [ ]:
if TENSEAL_AVAILABLE and ctx is not None:
    print("DEMOSTRACION DE OPERACIONES HOMOMORFICAS")
    print("=" * 60)
    
    # Datos de ejemplo
    a = [1.5, 2.5, 3.5]
    b = [0.5, 1.0, 1.5]
    
    print(f"\nDatos originales:")
    print(f"  a = {a}")
    print(f"  b = {b}")
    
    # Encriptar
    a_enc = ts.ckks_vector(ctx, a)
    b_enc = ts.ckks_vector(ctx, b)
    
    # Suma homomorfica
    sum_enc = a_enc + b_enc
    sum_dec = sum_enc.decrypt()
    sum_plain = [a[i] + b[i] for i in range(len(a))]
    
    print(f"\n[SUMA] a + b:")
    print(f"  Plaintext: {sum_plain}")
    print(f"  FHE result: {[round(x, 4) for x in sum_dec]}")
    
    # Multiplicacion homomorfica
    mul_enc = a_enc * b_enc
    mul_dec = mul_enc.decrypt()
    mul_plain = [a[i] * b[i] for i in range(len(a))]
    
    print(f"\n[MULTIPLICACION] a * b:")
    print(f"  Plaintext: {mul_plain}")
    print(f"  FHE result: {[round(x, 4) for x in mul_dec]}")
    
    # Producto escalar (dot product)
    weights = [0.3, 0.5, 0.2]
    weighted_sum_enc = a_enc.dot(weights)
    weighted_sum_dec = weighted_sum_enc.decrypt()[0]
    weighted_sum_plain = sum(a[i] * weights[i] for i in range(len(a)))
    
    print(f"\n[DOT PRODUCT] a . weights:")
    print(f"  Weights: {weights}")
    print(f"  Plaintext: {weighted_sum_plain:.4f}")
    print(f"  FHE result: {weighted_sum_dec:.4f}")
    
    print("\n" + "=" * 60)
    print("Las operaciones sobre datos cifrados producen resultados correctos!")
else:
    print("TenSEAL no disponible para demostrar operaciones")

## 6. Regresion Logistica sobre Datos Cifrados

Implementamos una version simplificada de inferencia con regresion logistica.

In [ ]:
class FHELogisticRegression:
    """
    Regresion Logistica simplificada para FHE.
    
    Nota: El entrenamiento se hace en plaintext, pero la inferencia
    puede hacerse sobre datos cifrados.
    """
    
    def __init__(self):
        self.weights = None
        self.bias = None
        self.sklearn_model = None
        
    def fit(self, X, y):
        """Entrenar modelo (en plaintext)."""
        self.sklearn_model = LogisticRegression(
            max_iter=1000,
            random_state=42
        )
        self.sklearn_model.fit(X, y)
        self.weights = self.sklearn_model.coef_[0].tolist()
        self.bias = self.sklearn_model.intercept_[0]
        
    def predict_plaintext(self, X):
        """Prediccion en plaintext."""
        return self.sklearn_model.predict(X)
    
    def predict_proba_plaintext(self, X):
        """Probabilidades en plaintext."""
        return self.sklearn_model.predict_proba(X)[:, 1]
    
    def predict_encrypted(self, X_encrypted, context):
        """
        Prediccion sobre datos cifrados.
        
        Calcula: z = X . weights + bias
        La funcion sigmoide se aproxima con polinomio.
        """
        if X_encrypted is None or context is None:
            return None
        
        predictions = []
        for x_enc in X_encrypted:
            # Producto punto con pesos
            z = x_enc.dot(self.weights)
            # Sumar bias
            z = z + self.bias
            # Desencriptar para obtener resultado
            z_dec = z.decrypt()[0]
            # Aplicar sigmoide (en plaintext para simplificar)
            prob = 1 / (1 + np.exp(-z_dec))
            predictions.append(1 if prob > 0.5 else 0)
            
        return np.array(predictions)

print("Clase FHELogisticRegression definida")

In [ ]:
# Entrenar modelo
print("ENTRENAMIENTO DEL MODELO")
print("=" * 60)

model = FHELogisticRegression()

print("Entrenando modelo en plaintext...")
start_time = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Entrenamiento completado en {train_time:.2f} segundos")

# Metricas en plaintext
y_pred_plain = model.predict_plaintext(X_test)
y_prob_plain = model.predict_proba_plaintext(X_test)

acc_plain = accuracy_score(y_test, y_pred_plain)
auc_plain = roc_auc_score(y_test, y_prob_plain)

print(f"\nResultados en PLAINTEXT:")
print(f"  Accuracy: {acc_plain*100:.2f}%")
print(f"  AUC-ROC: {auc_plain:.4f}")

In [ ]:
# Inferencia sobre datos cifrados
if TENSEAL_AVAILABLE and ctx is not None:
    print("\nINFERENCIA SOBRE DATOS CIFRADOS")
    print("=" * 60)
    
    # Encriptar datos de test
    print("Encriptando datos de test...")
    start_time = time.time()
    X_test_enc = encrypt_data(ctx, X_test)
    enc_time = time.time() - start_time
    print(f"  Tiempo de encriptacion: {enc_time:.2f} segundos")
    
    # Prediccion
    print("\nRealizando predicciones sobre ciphertext...")
    start_time = time.time()
    y_pred_fhe = model.predict_encrypted(X_test_enc, ctx)
    pred_time = time.time() - start_time
    print(f"  Tiempo de prediccion FHE: {pred_time:.2f} segundos")
    
    # Metricas FHE
    acc_fhe = accuracy_score(y_test, y_pred_fhe)
    
    print(f"\nResultados en FHE (CIFRADO):")
    print(f"  Accuracy: {acc_fhe*100:.2f}%")
    
    # Comparacion
    print("\n" + "=" * 60)
    print("COMPARACION PLAINTEXT vs FHE")
    print("-" * 40)
    print(f"{'Metrica':<20} {'Plaintext':>15} {'FHE':>15}")
    print("-" * 40)
    print(f"{'Accuracy':<20} {acc_plain*100:>14.2f}% {acc_fhe*100:>14.2f}%")
    print(f"{'Tiempo predict':<20} {'< 0.01s':>15} {pred_time:>14.2f}s")
    print("-" * 40)
    
    # Verificar que las predicciones coinciden
    matching = (y_pred_plain == y_pred_fhe).sum()
    print(f"\nPredicciones coincidentes: {matching}/{len(y_test)} ({matching/len(y_test)*100:.1f}%)")
else:
    print("\nInferencia FHE no disponible - TenSEAL no instalado")

## 7. Benchmark de Rendimiento

In [ ]:
if TENSEAL_AVAILABLE and ctx is not None:
    print("BENCHMARK DE RENDIMIENTO FHE")
    print("=" * 60)
    
    # Benchmark de encriptacion
    n_samples_bench = 100
    X_bench = X_test[:n_samples_bench]
    
    print(f"\nBenchmark con {n_samples_bench} muestras:")
    
    # Encriptacion
    start = time.time()
    for row in X_bench:
        _ = ts.ckks_vector(ctx, row.tolist())
    enc_time = time.time() - start
    print(f"  Encriptacion: {enc_time:.3f}s ({enc_time/n_samples_bench*1000:.2f} ms/muestra)")
    
    # Operacion (dot product)
    test_vec = ts.ckks_vector(ctx, X_bench[0].tolist())
    weights = model.weights
    
    start = time.time()
    for _ in range(n_samples_bench):
        _ = test_vec.dot(weights)
    op_time = time.time() - start
    print(f"  Dot product: {op_time:.3f}s ({op_time/n_samples_bench*1000:.2f} ms/operacion)")
    
    # Desencriptacion
    result_enc = test_vec.dot(weights)
    start = time.time()
    for _ in range(n_samples_bench):
        _ = result_enc.decrypt()
    dec_time = time.time() - start
    print(f"  Desencriptacion: {dec_time:.3f}s ({dec_time/n_samples_bench*1000:.2f} ms/muestra)")
    
    print("\nResumen de overhead FHE:")
    print(f"  Overhead total por prediccion: ~{(enc_time + op_time + dec_time)/n_samples_bench*1000:.0f} ms")
    print(f"  Factor de expansion de datos: ~100x")
else:
    print("Benchmark no disponible - TenSEAL no instalado")

## 8. Integracion con SDK de Xcapit

In [ ]:
# Intentar cargar SDK real
try:
    sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd()))))
    from sdk.encryption import CKKSEncryptor, SecurityLevel
    from sdk.models import LogisticRegression as FHELogReg
    SDK_AVAILABLE = True
    print("SDK de Xcapit cargado correctamente")
except ImportError:
    SDK_AVAILABLE = False
    print("SDK de Xcapit no disponible - usando implementacion local")

if SDK_AVAILABLE:
    print("\nUSO DEL SDK OFICIAL")
    print("=" * 60)
    print("""
    # Ejemplo de uso con SDK real:
    
    from sdk.encryption import CKKSEncryptor, SecurityLevel
    from sdk.models import LogisticRegression
    
    # Crear encriptador
    encryptor = CKKSEncryptor(security_level=SecurityLevel.BITS_128)
    
    # Encriptar datos
    X_enc = encryptor.encrypt(X_train)
    y_enc = encryptor.encrypt(y_train)
    
    # Entrenar modelo sobre datos cifrados
    model = LogisticRegression(encryptor=encryptor)
    model.fit(X_enc, y_enc, epochs=10)
    
    # Predecir
    predictions_enc = model.predict(X_test_enc)
    
    # Desencriptar resultados
    predictions = encryptor.decrypt(predictions_enc)
    """)

## 9. Resumen y Conclusiones

In [ ]:
print("RESUMEN DEL DEMO SDK FHE REAL")
print("=" * 60)

print("""
CARACTERISTICAS DE FHE (CKKS)
-----------------------------
 Permite operaciones sobre datos cifrados
 Los datos NUNCA se desencriptan en el servidor
 Solo el propietario de la clave puede ver resultados
 Seguridad basada en problemas matematicos dificiles (RLWE)

TRADE-OFFS
----------
  Overhead computacional: ~100-1000x mas lento que plaintext
  Expansion de datos: ~100x mas espacio
  Precision: Aproximada (CKKS usa aritmetica de punto flotante)

CASOS DE USO IDEALES
--------------------
 Datos altamente sensibles (medicos, financieros)
 Cumplimiento regulatorio estricto (GDPR, HIPAA)
 Colaboracion entre competidores
 Inferencia en la nube sin confianza
""")

if TENSEAL_AVAILABLE:
    print("\nMETRICAS DEL DEMO")
    print("-" * 40)
    print(f"  Muestras procesadas: {len(X_test)}")
    print(f"  Features: {N_FEATURES}")
    print(f"  Accuracy plaintext: {acc_plain*100:.2f}%")
    print(f"  Accuracy FHE: {acc_fhe*100:.2f}%")
    print(f"  Seguridad: 128 bits")

print("""
DOCUMENTACION
-------------
API: https://apifhe.xcapit.com/api/v2/docs/
SDK: https://github.com/xcapit/fhe-ml-sdk
TenSEAL: https://github.com/OpenMined/TenSEAL
""")

---

## Proximos Pasos

1. **Instalar TenSEAL**: `pip install tenseal`
2. **Probar con datos reales**: Conectar con la API de Xcapit
3. **Explorar otros modelos**: Decision Trees, Neural Networks
4. **Optimizar rendimiento**: Batch processing, GPU acceleration

**Documentacion completa**: https://apifhe.xcapit.com/api/v2/docs/